# 1. Train and test the cell-type-specific CNN-CR model of one cell line (Fig. 1d)

CNN-CR predicts a gene's expression from its regulatory windows and, while doing so, learns transcription-factor motifs. This notebook trains and tests the model on **K562 (EpiMap biosample BSS00762)** from the shipped window FASTA: the input files -> one-hot tensor -> gene split -> training -> the held-out predictions of Fig. 1d (and the per-biosample numbers behind Fig. 1c). Needs a GPU; the K562 input ships in the repository (`data/k562_demo/`, 20 MB), nothing to download.

The paper trained **ten models per biosample** (ten folds of the training genes, one held-out test set); `N_MODELS = 1` trains one (about 85 min on an A6000) and `N_MODELS = 10` reproduces the paper's protocol. Any other biosample can be run by changing `BSSID` once its input pair is unpacked from Zenodo `fig1_training_inputs_all116.tar.gz`. Before starting: the environment (`environment.yml`) and `bin/rewrite_paths.py` (see `PATHS.md`).

In [ ]:
# ---- Shared setup (identical in every notebook) --------------------------------------------------------
import os, sys, json, subprocess, time, glob, shutil
REPO = os.path.abspath(os.path.join(os.getcwd(), ".."))          # notebooks/ lives one level below the repository root
CODE = os.path.join(REPO, "code")
roots = json.load(open(os.path.join(REPO, ".roots.json")))          # written by bin/rewrite_paths.py (see PATHS.md)
ENHANCER_ROOT, SEQ_ROOT = roots["enhancer_root"], roots["seq_root"]
def run(cmd, cwd=None, env=None):
    """Run one repository script from the right folder and stream its output here. Stops the notebook if it fails."""
    print("$", cmd, "   (in", cwd or os.getcwd(), ")"); t = time.time()
    p = subprocess.Popen(cmd, shell=True, cwd=cwd, executable="/bin/bash", stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, env=env)
    for line in p.stdout: print(line, end="")
    p.wait(); print(f"[exit {p.returncode}, {time.time()-t:.0f} s]")
    assert p.returncode == 0, "this step failed - read the output above"
def n_motifs(meme):
    """Number of MOTIF entries in a MEME file."""
    return sum(1 for l in open(meme) if l.startswith("MOTIF")) if os.path.exists(meme) else None
def show_logos(meme, n=5, title=""):
    """Sequence logos of the first n motifs of a MEME file (logomaker)."""
    import numpy as np, pandas as pd, matplotlib.pyplot as plt, logomaker
    blocks = open(meme).read().split("MOTIF ")[1:n+1]; fig, axes = plt.subplots(1, len(blocks), figsize=(2.2*len(blocks), 1.4))
    for ax, b in zip(np.atleast_1d(axes), blocks):
        rows = [l.split() for l in b.split("\n")[2:] if l.strip() and not l.startswith("letter") and not l.startswith("URL")]
        pwm = pd.DataFrame([[float(x) for x in r] for r in rows if len(r) == 4], columns=list("ACGT"))
        logomaker.Logo(logomaker.transform_matrix(pwm, from_type="probability", to_type="information"), ax=ax); ax.set_title(b.split("\n")[0], fontsize=8); ax.set_yticks([]); ax.set_xticks([])
    fig.suptitle(title, fontsize=9); plt.tight_layout(); plt.show()
print("repository:", REPO); print("ENHANCER_ROOT:", ENHANCER_ROOT); print("SEQ_ROOT:", SEQ_ROOT)


In [ ]:
# ---- CONFIGURATION ---------------------------------------------------------------------------------------------
BSSID = "BSS00762"            # K562. Any of the 116 biosamples works once its two input files are in INPUTS (Zenodo fig1_training_inputs_all116)
N_MODELS = 1                  # models to train in section 1 (one per training fold); the paper trained 10
EPOCHS = 100                  # maximum epochs per model (paper: 100, early stopping with patience 10 on the validation fold)
SEED = 42
RUN_TRAINING = True           # False: skip training (no GPU) and use the released K562 fold-0 model and predictions in data/k562_sample
VALIDATE_EXTRACTION = True    # notebook 02, 1.3: extract from the released K562 fold-0 weights and compare with the released PWMs (10 min on a GPU)
RUN_TOMTOM_DEMO = True        # notebook 02, 3.3: run TOMTOM for the K562 promoter set against JASPAR here (about 4 min) and compare with the shipped table
INPUTS     = f"{ENHANCER_ROOT}/fig1_training_inputs" if BSSID != "BSS00762" else os.path.join(REPO, "data", "k562_demo")   # K562 ships in the repository; the other 115 biosamples: Zenodo fig1_training_inputs_all116 unpacked under ENHANCER_ROOT
MOTIF_SETS = f"{ENHANCER_ROOT}/fig1_motif_sets"         # Zenodo fig1_motif_sets_all116.tar.gz unpacked under ENHANCER_ROOT (section 4)
DEMO   = os.path.join(REPO, "data", "k562_demo")        # shipped: motifs + TOMTOM tables of the ten K562 models, K562 count matrices, K562 expression
SAMPLE = os.path.join(REPO, "data", "k562_sample")      # shipped: released K562 fold-0 model and the held-out predictions of the ten released models
WORK   = f"{ENHANCER_ROOT}/notebook_runs/fig1/{BSSID}"; os.makedirs(WORK, exist_ok=True)
sys.path.insert(0, f"{CODE}/cnncr_model"); sys.path.insert(0, f"{CODE}/motif_extraction")
import numpy as np, pandas as pd, matplotlib.pyplot as plt, torch, logomaker
from scipy.stats import spearmanr, pearsonr
from IPython.display import Image, display
from cnncr_io import read_windows_fasta, read_expression, one_hot      # code/cnncr_model/cnncr_io.py
import cnncr_model as cm                                                # code/cnncr_model/cnncr_model.py: the model, split and training loop of the paper
from extract_motifs_lib import extract_model, write_meme, read_meme     # code/motif_extraction/extract_motifs_lib.py
device = torch.device("cuda" if torch.cuda.is_available() else "cpu"); print("device:", device, "| work folder:", WORK)
PRO, PRO_F, ENH, ENH_F = "#8FB8C9", "#D6E7EE", "#D79581", "#F4E0D8"        # promoter / enhancer colours of the paper's figures
def show_pwms(pwms, titles, suptitle=""):
    """sequence logos (information content) of a few PWMs"""
    fig, axes = plt.subplots(1, len(pwms), figsize=(2.0*len(pwms), 1.3))
    for ax, p, t in zip(np.atleast_1d(axes), pwms, titles):
        logomaker.Logo(logomaker.transform_matrix(pd.DataFrame(p, columns=list("ACGT")), from_type="probability", to_type="information"), ax=ax)
        ax.set_title(t, fontsize=8); ax.set_yticks([]); ax.set_xticks([])
    fig.suptitle(suptitle, fontsize=9); plt.tight_layout(); plt.show()
def spearman(a, b): return spearmanr(a, b)[0]
def same_pwms(a, b):
    """the same PWMs regardless of order (each PWM rounded to 6 decimals)"""
    key = lambda m: np.round(m, 6).tobytes(); ka, kb = set(map(key, a)), set(map(key, b))
    return f"{len(ka & kb)} of {len(kb)} released PWMs recovered" + (" (identical set)" if len(a) == len(b) and ka == kb else "")

## 1. From the window FASTA to Fig. 1d

### 1.1 The input: 400-bp windows and log2 expression
Each biosample's input is one FASTA of regulatory windows and one expression table (K562: `data/k562_demo/`; all 116 biosamples: Zenodo `fig1_training_inputs_all116`). Every gene has up to twenty 400-bp windows: slot 0 is the TSS ± 200 bp (every gene has it), slots 1-9 are up to nine promoter-proximal peak windows within TSS ± 10 kb, slots 10-19 up to ten EpiTensor-linked enhancer windows. A slot the gene does not have is absent from the file. The target is log2 FPKM of the same 18,356 genes.

In [ ]:
print(open(f"{INPUTS}/README_training_inputs.txt" if BSSID == "BSS00762" else f"{INPUTS}/README.txt").read())
fa, ex = f"{INPUTS}/{BSSID}.windows.fa.gz", f"{INPUTS}/{BSSID}.expression.tsv"
X, headers = read_windows_fasta(fa, n_genes=18356)      # (genes, 20 slots, 400 bp, ACGT) bool
genes, y = read_expression(ex)
print("first three window headers:", *headers[:3], sep="\n  ")
print(f"\nX {X.shape} {X.dtype} | y {y.shape}: log2 FPKM, range {y.min():.2f} to {y.max():.2f}, median {np.median(y):.2f}")
present = X.any(axis=(2, 3))                                                          # genes x 20: which slots exist
fig, ax = plt.subplots(1, 2, figsize=(8.5, 2.4))
ax[0].bar(range(20), present.mean(0) * 100, color=[PRO]*10 + [ENH]*10); ax[0].set_xlabel("window slot (0-9 promoter, 10-19 enhancer)"); ax[0].set_ylabel("% genes with a window")
ax[1].hist(y, bins=60, color="#7E86B8"); ax[1].set_xlabel("log2 FPKM"); ax[1].set_ylabel("genes"); plt.tight_layout(); plt.show()
print(f"windows per gene: {present[:, :10].sum(1).mean():.2f} promoter, {present[:, 10:].sum(1).mean():.2f} enhancer")

### 1.2 One-hot encoding
A window becomes a 400 × 4 matrix with one 1 per position (A, C, G, T); N and absent windows are all zeros. `read_windows_fasta` did this for every window while reading, so `X` is already the model tensor; below is the encoding on a short example and on the first window of the file.

In [ ]:
seq = "ACGTNACGT"
print(seq); print(pd.DataFrame(one_hot(seq).astype(int).T, index=list("ACGT"), columns=list(seq)))
print("\nfirst window of the file,", headers[0], "- first 6 bp:"); print(X[0, 0, :6].astype(int))

### 1.3 The gene split
The genes are permuted (seeded here; the released models used an unseeded permutation, so their exact held-out genes are those recorded in their `_test_y_all.npy`), 10 % are held out as the test set (`random_state = 42`), and the remaining 90 % are split into ten folds: model *i* trains on nine folds and uses fold *i* for early stopping. `cross_val_gene` is the paper's split function.

In [ ]:
perm = np.random.RandomState(SEED).permutation(len(y)); np.save(f"{WORK}/{BSSID}_gene_permutation.npy", perm)
X_enc, y_enc = np.stack([X[perm]]), np.stack([y[perm]])          # the helper expects a leading axis (it was written for several biosamples at once)
Xtr, ytr, Xva, yva, Xte, yte = cm.cross_val_gene(X_enc, y_enc, 0)
print("fold 0: train", ytr.size, "genes | validation (early stopping)", yva.size, "| held-out test", yte.size)
del Xtr, Xva, Xte

### 1.4 The model and its training
`MultiBranchModel(num_branches=20)`: twenty branches (one per window slot) that do not share weights. In each branch 256 filters of 8 bp scan the window (the motif detectors), a small network computes a **contextual weight** per filter from the same window, and the branch output is the sum over filters of contextual weight × mean filter activation; the twenty branch outputs are summed into the predicted log2 expression. Training: Adam (lr 10⁻⁴), MSE plus L2 on the contextual-weight layers, batch 64, early stopping on the validation fold (patience 10). All of this is `cnncr_model.py`, the paper's code.

In [ ]:
from torch.utils.data import DataLoader, TensorDataset
def train_fold(iteration):
    """train model `iteration` on its nine training folds, early-stop on fold `iteration`, predict the held-out test genes"""
    Xtr, ytr, Xva, yva, Xte, yte = cm.cross_val_gene(X_enc, y_enc, iteration)
    Xtr, Xva, Xte = (a.reshape(-1, *a.shape[2:]) for a in (Xtr, Xva, Xte)); ytr, yva, yte = ytr.flatten(), yva.flatten(), yte.flatten()
    T = lambda z: torch.tensor(np.asarray(z)).float()
    model = cm.MultiBranchModel(num_branches=20).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=cm.lr, weight_decay=cm.decay)
    train_loader = DataLoader(TensorDataset(T(Xtr), T(ytr).unsqueeze(1)), batch_size=cm.batch, shuffle=True)
    val_loader   = DataLoader(TensorDataset(T(Xva), T(yva).unsqueeze(1)), batch_size=len(yva), shuffle=False)
    model, train_loss, val_loss = cm.train_model(model, train_loader, val_loader, optimizer, torch.nn.MSELoss(), EPOCHS, device)
    torch.save(model.state_dict(), f"{WORK}/{BSSID}_it_{iteration}_256MOTIF_pytorch.pth")
    pred = cm.predict(model, T(Xte), device).numpy().flatten()
    return model, pred, yte, train_loss, val_loss
if RUN_TRAINING:
    torch.manual_seed(SEED); models, preds, losses = [], [], []
    for it in range(N_MODELS):
        t0 = time.time(); model, pred, yte, tl, vl = train_fold(it); models.append(model); preds.append(pred); losses.append((tl, vl))
        print(f"model {it}: {len(tl)} epochs, {(time.time()-t0)/60:.1f} min | held-out Spearman {spearman(yte, pred):.3f}, Pearson {pearsonr(yte, pred)[0]:.3f}\n")
    np.save(f"{WORK}/{BSSID}256MOTIF_pytorch_test_pred_all.npy", np.array(preds)); np.save(f"{WORK}/{BSSID}256MOTIF_pytorch_test_y_all.npy", np.array([yte] * len(preds)))
    fig, ax = plt.subplots(figsize=(4, 2.2))
    for it, (tl, vl) in enumerate(losses): ax.plot(tl, color=PRO, lw=1, label="train" if it == 0 else None); ax.plot(vl, color=ENH, lw=1, label="validation" if it == 0 else None)
    ax.set_xlabel("epoch"); ax.set_ylabel("MSE (+ L2)"); ax.legend(frameon=False); plt.tight_layout(); plt.show()
    PRED_DIR = WORK
else:
    print("RUN_TRAINING = False: using the released K562 models and predictions in data/k562_sample"); PRED_DIR = SAMPLE

### 1.5 Fig. 1d: predicted versus measured expression of the held-out genes
The panel shows, for the 1,836 held-out K562 genes, the median prediction of the ten models (here: of the `N_MODELS` you trained) against the measured expression, on a log10 axis. `figures/fig1/make_fig1d_k562.py` draws it at the paper's size from any folder holding `*_test_pred_all.npy` / `*_test_y_all.npy`. The second panel is the published one, from the held-out predictions of the ten released models (`data/k562_sample/`, Spearman ρ = 0.806). Panel 1c of the paper is the same held-out Spearman/Pearson for every biosample (median ρ = 0.74 over 116 biosamples; 0.743 vs 0.735 for the full vs promoter-only model over the 79 biosamples with both).

In [ ]:
run(f"python {REPO}/figures/fig1/make_fig1d_k562.py --pred_dir {PRED_DIR} --bssid {BSSID} --label {'K562' if BSSID == 'BSS00762' else BSSID} --out {WORK}/fig1d_{BSSID}", cwd=WORK)
run(f"python {REPO}/figures/fig1/make_fig1d_k562.py --pred_dir {SAMPLE} --bssid BSS00762 --label 'K562 (released)' --out {WORK}/fig1d_released", cwd=WORK)
display(Image(f"{WORK}/fig1d_{BSSID}.png", width=300)); display(Image(f"{WORK}/fig1d_released.png", width=300))
rel = np.load(f"{SAMPLE}/BSS00762256MOTIF_pytorch_test_corr_all_S.npy"); print("released K562 models, held-out Spearman per model:", np.round(rel, 3), "median %.3f" % np.median(rel))